# Phase 4: Category Models and Thresholds
English training data: 500k natural comments (learning curve: natural > enriched for
toxicity) plus all comments from rare categories (needed for threat etc.).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from portfolio_projekt.data import DATA_DIR, LABELS, THRESHOLD, load_english, load_german

In [ ]:
# Validation and test stay unchanged; German data is used as prepared in Phase 2
_, df_en_val, df_en_test = load_english()
df_de_train, df_de_val, df_de_test = load_german()

# Full English training split without texts that appear in validation or test
df_en_full = pd.read_parquet(DATA_DIR / "raw" / "civil_comments" / "train")
in_eval = df_en_full["text"].isin(df_en_val["text"]) | df_en_full["text"].isin(df_en_test["text"])
df_en_full = df_en_full[~in_eval].drop_duplicates(subset="text")

# 500k natural comments plus all comments from rare categories
rare_labels = ["obscene", "threat", "identity_attack", "sexual_explicit"]
is_rare = (df_en_full[rare_labels] >= THRESHOLD).any(axis=1)

df_en_train = pd.concat([
    df_en_full.sample(500_000, random_state=42),
    df_en_full[is_rare],
]).drop_duplicates(subset="text")

del df_en_full
print(df_en_train.shape)
(df_en_train[LABELS] >= THRESHOLD).mean().mul(100).round(2)